<a href="https://colab.research.google.com/github/DenysovOleksii/AB_TEST/blob/main/AB_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [133]:
import pandas as pd
from scipy import stats
import numpy as np
from google.colab import drive
import statsmodels.api as sm
drive.mount("/drive/")


Mounted at /drive/


In [134]:
df=pd.read_csv("/drive/MyDrive/bq-results-20260522-143350-1779460502730/bq-results-20260522-143350-1779460502730.csv")

In [135]:
df['event_name'].value_counts().loc["add_payment_info"]/df['event_name'].value_counts().loc["session"]

np.float64(0.07506762428877903)

In [136]:
# СТАТИЧНІ КОНСТАНТИ (визначені завданням метрики)
METRIC_CONVERSIONS = [
    "add_payment_info",
    "add_shipping_info",
    "begin_checkout",
    "new account",
]
BASE_VOLUME = "sessions"


In [137]:
df_aggregated=df.groupby(["test","test_group","event_name"])["value"].sum().unstack()

In [138]:
def generate_ab_test_csv(data, tests=[1, 2, 3, 4], metrics=["add_payment_info", "add_shipping_info", "begin_checkout", "new account"], alpha=0.05):
    all_rows = []

    for test in tests:
        # Check if the test number exists in the data to avoid KeyErrors
        if test not in data.index.get_level_values(0):
            continue

        df_data = data.loc[test]

        # Ensure both control (1) and test (2) groups are present
        if 1 not in df_data.index or 2 not in df_data.index:
            continue

        # Denominator (total number of sessions) for both groups
        nobs_control = df_data.loc[1, "session"]
        nobs_test = df_data.loc[2, "session"]

        for metric in metrics:
            if metric not in df_data.columns:
                continue

            # Numerator (total number of target events)
            count_control = df_data.loc[1, metric]
            count_test = df_data.loc[2, metric]

            # Calculate conversion rates (CR)
            cr_control = count_control / nobs_control
            cr_test = count_test / nobs_test

            # Calculate relative metric change in %
            metric_change = (cr_control - cr_test) / cr_control * 100

            # Run the Z-test for proportions
            z_stat, p_value = sm.stats.proportions_ztest(
                [count_control, count_test],
                [nobs_control, nobs_test]
            )


            # Determine statistical significance
            significant = "TRUE" if p_value < alpha else "FALSE"

            # Construct the structured data row
            row = {
                "test_number": test,
                "metric": metric,
                "numerator_event": metric,
                "denominator_event": "session",
                "numerator_control": count_control,
                "denominator_control": nobs_control,
                "conversion_rate_control": cr_control,
                "numerator_test": count_test,
                "denominator_test": nobs_test,
                "conversion_rate_test": cr_test,
                "metric_change_percent": metric_change,
                "z_stat": z_stat,
                "p_value": p_value,
                "significant": significant
            }
            all_rows.append(row)

    # Combine all records into a final DataFrame
    result_df = pd.DataFrame(all_rows)
    return result_df

In [139]:
generate_ab_test_csv(df_aggregated).to_csv("/drive/MyDrive/ab_test.csv", index=False)

In [140]:
generate_ab_test_csv(df_aggregated)

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change_percent,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,-12.542021,-3.924884,0.000087,TRUE
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,-6.560481,-2.603571,0.009226,TRUE
2,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,-6.660587,-2.978783,0.002894,TRUE
3,1,new account,new account,session,3823,45362,0.084278,3681,45193,0.081451,3.354299,1.542883,0.122859,FALSE
4,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,-3.576911,-1.240994,0.214608,FALSE
5,2,add_shipping_info,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,-1.650995,-0.709557,0.477979,FALSE
6,2,begin_checkout,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,-1.988164,-0.952898,0.340642,FALSE
7,2,new account,new account,session,4165,50637,0.082252,4184,50244,0.083274,-1.241934,-0.588793,0.556000,FALSE
8,3,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,-1.474630,-0.643172,0.520112,FALSE
9,3,add_shipping_info,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,2.621211,1.413727,0.157442,FALSE
